# Prometheus Star: ARC-AGI-3 Solver (Production Edition)
**Architecture:** Bridge v15 (Neural Latent Reasoning)
**Last updated:** 2026-04-09

This notebook is streamlined for solving the official ARC-AGI-3 benchmark using the latest repository updates.

In [ ]:
# ── Colab / local setup ──────────────────────────────────────────────────────
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print('Cloning Prometheus repository...')
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    else:
        os.system('git -C Prometheus_v0_PoC pull origin wp16-notebook-only')
    
    print('Installing dependencies...')
    os.system('pip install -q arc-agi')
    os.system('pip install -q -e Prometheus_v0_PoC/')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, '..')

import json
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

# ── WP71 imports ───────────
from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer, ARC3ExplorationPolicy,
    ARC3StrangeLoopAgent, ARC3Benchmark, ARC3BenchmarkReport,
    verify_wp71_exit_criteria, _SyntheticARCGame, _ACTION_TYPES,
)

print('WP71 ARC-AGI-3 module loaded.')
print(f'Canonical action types ({len(_ACTION_TYPES)}): {_ACTION_TYPES}')

### Step 2: API Configuration
To access the full ARC-AGI-3 dataset and benchmark, you need an API key from **[three.arcprize.org](https://three.arcprize.org)**.

1. Log in to [three.arcprize.org](https://three.arcprize.org)
2. Copy your **API Key**
3. In Colab, click the **Secrets** (key icon) on the left sidebar
4. Add a new secret with name `ARC_API_KEY` and paste your key as the value
5. Enable the **Notebook access** toggle for this secret

In [ ]:
# Cell 2: API Configuration & Bridge Loading
import os
try:
    from google.colab import userdata
    ARC_API_KEY = userdata.get("ARC_API_KEY")
    os.environ["ARC_API_KEY"] = ARC_API_KEY
    if ARC_API_KEY:
        print(f"API Key active ({ARC_API_KEY[:6]}...)")
    else:
        print("No API Key found - anonymous access enabled.")
except:
    ARC_API_KEY = None
    print("No API Key found - anonymous access enabled.")

from prometheus.arc3_bridge import *
print(f"Bridge v15 loaded (Toolkit Available: {TOOLKIT_AVAILABLE}).")

def visualize_arc3_episode(episode, max_steps=10):
    """Visualise the first N steps of an ARC-AGI-3 episode."""
    steps = min(len(episode.history), max_steps)
    if steps == 0: return
    
    fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2))
    if steps == 1: axes = [axes]
    
    for i in range(steps):
        obs, action, reward = episode.history[i]
        grid = np.array(obs.grid)
        axes[i].imshow(grid, cmap='tab20', vmin=0, vmax=15)
        axes[i].set_title(f"S{i}: {action.action_type}\nR={reward:.1f}", fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Run Prometheus on ARC-AGI-3 ──────
#
# Bridge v15: Transformer World Model + Active Hypothesis Testing.

SELECTED_GAMES = ["ls20", "ft09", "vc33"]
if ARC_API_KEY and TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        all_envs = arc.get_environments()
        SELECTED_GAMES = [e.game_id for e in all_envs]
        print(f"API Key detected: Loading all {len(SELECTED_GAMES)} games.")
    except:
        print("API Key failed: Falling back to public games.")

N_WINDOWS     = 60
WINDOW_STEPS  = 200

live_results = {}
t_start = time.time()

# Run on first 5 games to keep demo snappy if all loaded
GAMES_TO_RUN = SELECTED_GAMES[:5] if len(SELECTED_GAMES) > 3 else SELECTED_GAMES

for game_id in GAMES_TO_RUN:
    print()
    print('=' * 55)
    print(f"  Game: {game_id}")
    print('=' * 55)
    result = run_live_game(game_id=game_id, n_windows=N_WINDOWS, window_steps=WINDOW_STEPS, mutation_rate=0.10, fitness_threshold=0.5, verbose=True)
    if result:
        live_results[game_id] = result
        sr_val = result.get('solve_rate', 0)
        ms_val = result.get('mean_score', 0)
        print(f"  --> solve rate: {sr_val:.0%}  mean score: {ms_val:.3f}")
        
        # Visualize the last episode
        if result.get('last_episode'):
            print(f"  Visualising last window of {game_id}...")
            visualize_arc3_episode(result['last_episode'], max_steps=8)

print()
print(f"Total time: {time.time()-t_start:.1f}s")
